# 7. N-Step Bootstrapping

**Kaynak:** Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd Edition (2018)
- **Bölüm 7: n-step Bootstrapping** (Sayfa 147-162)

## İçindekiler
1. N-Step TD Prediction *(s. 147-152)*
2. N-Step SARSA *(s. 153-155)*
3. N-Step Off-policy Learning *(s. 155-158)*
4. N-Step Tree Backup *(s. 158-160)*

---
## 7.1 TD ve MC Arasındaki Spektrum

📖 **Referans:** Sutton & Barto, Sayfa 147-149, Section 7.1

> *"n-step TD methods span a spectrum with MC methods at one end and one-step TD methods at the other."* (s. 147)

TD(0) ve Monte Carlo, bir spektrumun iki ucudur:

| Yöntem | Kaç adım bekler? | Bootstrap? |
|--------|-----------------|------------|
| TD(0) | 1 adım | Evet |
| 2-step TD | 2 adım | Evet |
| n-step TD | n adım | Evet |
| MC | Episode sonu | Hayır |

### N-Step Return - Equation 7.1 (s. 148)

$$G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V_{t+n-1}(S_{t+n})$$

> *"The target for the one-step update is the first reward plus the discounted estimated value of the next state... the target for the two-step update is the first two rewards plus the discounted estimated value of the state two steps later"* (s. 148)

- $n=1$: $G_{t:t+1} = R_{t+1} + \gamma V(S_{t+1})$ → TD(0)
- $n=\infty$: $G_{t:\infty} = G_t$ → MC

In [ ]:
# Kod Örneği: Random Walk Environment
# Referans: Example 7.1 (s. 150) - "19-state Random Walk"

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

class RandomWalkEnv:
    """
    19-state Random Walk.
    Referans: Example 7.1 (s. 150)
    
    "Consider the 19-state random walk task... The agent starts in the 
    center state and then moves randomly left or right until reaching 
    one of the two terminal states."
    
    States: 0 (left terminal), 1-19, 20 (right terminal)
    Start: state 10 (middle)
    Left terminal: reward -1
    Right terminal: reward +1
    """
    
    def __init__(self, n_states=19):
        self.n_states = n_states
        self.start_state = n_states // 2 + 1  # Middle state
        self.left_terminal = 0
        self.right_terminal = n_states + 1
        self.reset()
    
    def reset(self):
        self.state = self.start_state
        return self.state
    
    def step(self, action=None):
        """
        Random walk: 50% left, 50% right.
        "The left terminal state produces a reward of −1; the right 
        terminal state produces +1" (s. 150)
        """
        if np.random.random() < 0.5:
            self.state -= 1  # Left
        else:
            self.state += 1  # Right
        
        # Check terminals
        if self.state == self.left_terminal:
            return self.state, -1, True
        elif self.state == self.right_terminal:
            return self.state, 1, True
        else:
            return self.state, 0, False

# True values for random walk
def compute_true_values(n_states=19):
    """True state values for random walk."""
    return np.array([(2*s - n_states - 1) / (n_states + 1) for s in range(1, n_states + 1)])

env = RandomWalkEnv()
true_values = compute_true_values()

print(f"States: 1-{env.n_states}")
print(f"Start state: {env.start_state}")
print(f"True values: {true_values.round(2)}")

In [ ]:
# Visualize random walk
plt.figure(figsize=(12, 3))
states = range(1, env.n_states + 1)

plt.bar(states, true_values, color='steelblue', alpha=0.7)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.xlabel('State')
plt.ylabel('True Value V(s)')
plt.title('19-State Random Walk: True Values')
plt.xticks(states)
plt.grid(True, alpha=0.3)
plt.show()

---
## 7.2 N-Step TD Prediction

📖 **Referans:** Sutton & Barto, Sayfa 149-152, Algorithm on page 150

> *"The natural state-value learning algorithm for using n-step returns is thus"* (s. 149)

### Update Rule - Equation 7.2 (s. 149)

$$V_{t+n}(S_t) = V_{t+n-1}(S_t) + \alpha [G_{t:t+n} - V_{t+n-1}(S_t)]$$

where (Equation 7.1, s. 148):
$$G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V_{t+n-1}(S_{t+n})$$

> *"Note that no changes at all are made during the first n − 1 steps of each episode. To make up for that, an equal number of additional updates are made at the end of the episode"* (s. 150)

In [ ]:
def n_step_td_prediction(env, n, n_episodes=10, alpha=0.1, gamma=1.0):
    """
    N-step TD Prediction.
    Referans: Algorithm (s. 150) - "n-step TD for estimating V ≈ v_π"
    
    Args:
        env: Environment
        n: Number of steps ("n is a positive integer")
        n_episodes: Number of episodes
        alpha: "step-size parameter α ∈ (0, 1]"
        gamma: Discount factor
    
    Returns:
        V: State value estimates
    """
    # "Initialize V(s) arbitrarily, for all s ∈ S"
    V = np.zeros(env.n_states + 2)
    
    # "Loop for each episode:"
    for episode in range(n_episodes):
        # "Initialize and store S_0 ≠ terminal"
        states = [env.reset()]
        rewards = [0]  # R_0 is not used
        
        T = float('inf')  # "T ← ∞"
        t = 0
        
        # "Loop for t = 0, 1, 2, ...:"
        while True:
            if t < T:
                # "Take action according to π"
                next_state, reward, done = env.step()
                states.append(next_state)  # "Store S_{t+1}"
                rewards.append(reward)  # "Store R_{t+1}"
                
                if done:  # "If S_{t+1} is terminal"
                    T = t + 1  # "T ← t + 1"
            
            # "τ ← t − n + 1 (τ is the time whose state's estimate is being updated)"
            tau = t - n + 1
            
            if tau >= 0:
                # "G ← Σ_{i=τ+1}^{min(τ+n,T)} γ^{i-τ-1} R_i" - Equation 7.1
                G = 0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                
                # "If τ + n < T, then G ← G + γ^n V(S_{τ+n})"
                if tau + n < T:
                    G += (gamma ** n) * V[states[tau + n]]
                
                # "V(S_τ) ← V(S_τ) + α[G − V(S_τ)]"
                s = states[tau]
                if 1 <= s <= env.n_states:
                    V[s] += alpha * (G - V[s])
            
            # "Until τ = T − 1"
            if tau == T - 1:
                break
            
            t += 1
    
    return V[1:env.n_states + 1]

# Test different n values - Figure 7.2 (s. 151)
n_values = [1, 2, 4, 8, 16]
n_episodes = 10

results = {}
for n in n_values:
    V = n_step_td_prediction(env, n, n_episodes=n_episodes, alpha=0.4)
    rmse = np.sqrt(np.mean((V - true_values) ** 2))
    results[n] = {'V': V, 'rmse': rmse}
    print(f"n={n}: RMSE = {rmse:.4f}")

In [ ]:
# Figure 7.2 (s. 151) - Effect of n and α on n-step TD
# "Performance of n-step TD methods as a function of α, for various values of n"

def run_n_step_experiment(env, n_values, alphas, n_episodes=10, n_runs=100):
    """
    Run experiments for different n and alpha values.
    Referans: Figure 7.2 (s. 151)
    """
    
    true_v = compute_true_values()
    results = np.zeros((len(n_values), len(alphas)))
    
    for i, n in enumerate(n_values):
        for j, alpha in enumerate(alphas):
            errors = []
            for _ in range(n_runs):
                V = n_step_td_prediction(env, n, n_episodes=n_episodes, alpha=alpha)
                rmse = np.sqrt(np.mean((V - true_v) ** 2))
                errors.append(rmse)
            results[i, j] = np.mean(errors)
    
    return results

n_values = [1, 2, 4, 8, 16, 32]
alphas = np.linspace(0.1, 1.0, 10)

print("Running experiments (Figure 7.2)...")
rmse_results = run_n_step_experiment(env, n_values, alphas, n_episodes=10, n_runs=50)

# Plot - Figure 7.2 (s. 151)
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(n_values)))

for i, n in enumerate(n_values):
    plt.plot(alphas, rmse_results[i], label=f'n={n}', color=colors[i], linewidth=2)

plt.xlabel('α')
plt.ylabel('Average RMS error over 19 states and first 10 episodes')
plt.title('N-Step TD: Figure 7.2 (s. 151)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 7.3 N-Step SARSA

📖 **Referans:** Sutton & Barto, Sayfa 153-155, Section 7.2

> *"Ideas of n-step methods are straightforward to extend to action-value methods... The main idea is to simply switch states for actions (state–action pairs) and then use an ε-greedy policy."* (s. 153)

N-step fikrini control'e uygula:

### N-Step Return (Q için) - Equation 7.4 (s. 153)

$$G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n Q_{t+n-1}(S_{t+n}, A_{t+n})$$

### Update - Equation 7.5 (s. 153)

$$Q_{t+n}(S_t, A_t) = Q_{t+n-1}(S_t, A_t) + \alpha [G_{t:t+n} - Q_{t+n-1}(S_t, A_t)]$$

In [ ]:
class GridWorldEnv:
    """Simple grid world for n-step SARSA demo."""
    
    def __init__(self, rows=4, cols=4):
        self.rows = rows
        self.cols = cols
        self.start = (3, 0)
        self.goal = (0, 3)
        self.n_actions = 4
        
        self.actions = {
            0: (-1, 0),  # up
            1: (0, 1),   # right
            2: (1, 0),   # down
            3: (0, -1)   # left
        }
        self.reset()
    
    def reset(self):
        self.position = self.start
        return self.position
    
    def step(self, action):
        row, col = self.position
        drow, dcol = self.actions[action]
        
        new_row = max(0, min(self.rows - 1, row + drow))
        new_col = max(0, min(self.cols - 1, col + dcol))
        self.position = (new_row, new_col)
        
        if self.position == self.goal:
            return self.position, 0, True
        return self.position, -1, False

env_grid = GridWorldEnv()

In [ ]:
def n_step_sarsa(env, n, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    N-step SARSA.
    Referans: Algorithm (s. 154) - "n-step Sarsa for estimating Q ≈ q_*"
    """
    # "Initialize Q(s, a) arbitrarily, for all s ∈ S, a ∈ A"
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def epsilon_greedy(state):
        """ε-greedy policy"""
        if np.random.random() < epsilon:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    episode_lengths = []
    
    # "Loop for each episode:"
    for episode in range(n_episodes):
        # "Initialize and store S_0 ≠ terminal"
        states = [env.reset()]
        # "Select and store an action A_0 ~ π(·|S_0)"
        actions = [epsilon_greedy(states[0])]
        rewards = [0]
        
        T = float('inf')  # "T ← ∞"
        t = 0
        
        # "Loop for t = 0, 1, 2, ...:"
        while True:
            if t < T:
                # "Take action A_t"
                next_state, reward, done = env.step(actions[t])
                states.append(next_state)  # "Store S_{t+1}"
                rewards.append(reward)  # "Store R_{t+1}"
                
                if done:  # "If S_{t+1} is terminal"
                    T = t + 1  # "T ← t + 1"
                else:
                    # "Select and store A_{t+1} ~ π(·|S_{t+1})"
                    actions.append(epsilon_greedy(next_state))
            
            # "τ ← t − n + 1"
            tau = t - n + 1
            
            if tau >= 0:
                # "G ← Σ_{i=τ+1}^{min(τ+n,T)} γ^{i-τ-1} R_i" - Equation 7.4
                G = 0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                
                # "If τ + n < T: G ← G + γ^n Q(S_{τ+n}, A_{τ+n})"
                if tau + n < T:
                    G += (gamma ** n) * Q[states[tau + n]][actions[tau + n]]
                
                # "Q(S_τ, A_τ) ← Q(S_τ, A_τ) + α[G − Q(S_τ, A_τ)]"
                s, a = states[tau], actions[tau]
                Q[s][a] += alpha * (G - Q[s][a])
            
            # "Until τ = T − 1"
            if tau == T - 1:
                break
            
            t += 1
        
        episode_lengths.append(T)
    
    return Q, episode_lengths

# Compare different n values
n_values = [1, 2, 4, 8]
results_sarsa = {}

for n in n_values:
    Q, lengths = n_step_sarsa(env_grid, n, n_episodes=200)
    results_sarsa[n] = lengths
    print(f"n={n}: Average episode length (last 50) = {np.mean(lengths[-50:]):.1f}")

In [ ]:
# Plot learning curves
plt.figure(figsize=(10, 5))

window = 10
for n in n_values:
    smoothed = np.convolve(results_sarsa[n], np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=f'n={n}', linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Steps per Episode')
plt.title('N-Step SARSA: Learning Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## 7.4 N-Step Off-policy Learning

📖 **Referans:** Sutton & Barto, Sayfa 155-158, Section 7.3

> *"Recall that off-policy learning is learning the value function for one policy, π, while following another policy, b."* (s. 155)

Off-policy n-step learning, importance sampling gerektirir:

### Importance Sampling Ratio - Equation 7.9 (s. 156)

$$\rho_{t:h} = \prod_{k=t}^{\min(h, T-1)} \frac{\pi(A_k|S_k)}{b(A_k|S_k)}$$

### N-Step Off-policy Update - Equation 7.9 (s. 156)

$$V_{t+n}(S_t) = V_{t+n-1}(S_t) + \alpha \rho_{t:t+n-1} [G_{t:t+n} - V_{t+n-1}(S_t)]$$

> *"The importance sampling ratio here starts one step later than for MC methods because here we are updating a state and do not have to care how it was selected."* (s. 156)

In [ ]:
def n_step_off_policy_sarsa(env, n, n_episodes=500, alpha=0.5, gamma=1.0, 
                            epsilon_behavior=0.1, epsilon_target=0.0):
    """
    N-step Off-policy SARSA with Importance Sampling.
    Referans: Algorithm (s. 157) - "n-step off-policy Q* estimation"
    
    behavior policy b: ε-greedy with epsilon_behavior
    target policy π: ε-greedy with epsilon_target (0 = greedy)
    """
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def behavior_policy(state):
        """b(a|s): Behavior policy"""
        if np.random.random() < epsilon_behavior:
            return np.random.randint(env.n_actions)
        return np.argmax(Q[state])
    
    def target_policy_prob(state, action):
        """π(a|s) for target policy."""
        best_action = np.argmax(Q[state])
        if action == best_action:
            return 1 - epsilon_target + epsilon_target / env.n_actions
        return epsilon_target / env.n_actions
    
    def behavior_policy_prob(state, action):
        """b(a|s) for behavior policy."""
        best_action = np.argmax(Q[state])
        if action == best_action:
            return 1 - epsilon_behavior + epsilon_behavior / env.n_actions
        return epsilon_behavior / env.n_actions
    
    episode_lengths = []
    
    for episode in range(n_episodes):
        states = [env.reset()]
        actions = [behavior_policy(states[0])]
        rewards = [0]
        
        T = float('inf')
        t = 0
        
        while True:
            if t < T:
                next_state, reward, done = env.step(actions[t])
                states.append(next_state)
                rewards.append(reward)
                
                if done:
                    T = t + 1
                else:
                    actions.append(behavior_policy(next_state))
            
            tau = t - n + 1
            
            if tau >= 0:
                # "ρ ← Π_{k=τ+1}^{min(τ+n-1,T-1)} π(A_k|S_k) / b(A_k|S_k)" - Equation 7.9
                rho = 1.0
                for k in range(tau + 1, min(tau + n, T)):
                    pi_prob = target_policy_prob(states[k], actions[k])
                    b_prob = behavior_policy_prob(states[k], actions[k])
                    rho *= pi_prob / b_prob
                
                # Calculate n-step return
                G = 0
                for i in range(tau + 1, min(tau + n, T) + 1):
                    G += (gamma ** (i - tau - 1)) * rewards[i]
                
                if tau + n < T:
                    G += (gamma ** n) * Q[states[tau + n]][actions[tau + n]]
                
                # "Q(S_τ,A_τ) ← Q(S_τ,A_τ) + αρ[G − Q(S_τ,A_τ)]"
                s, a = states[tau], actions[tau]
                Q[s][a] += alpha * rho * (G - Q[s][a])
            
            if tau == T - 1:
                break
            
            t += 1
        
        episode_lengths.append(T)
    
    return Q, episode_lengths

Q_off, lengths_off = n_step_off_policy_sarsa(env_grid, n=4, n_episodes=200)
print(f"Off-policy n-step SARSA: Avg length = {np.mean(lengths_off[-50:]):.1f}")

---
## 7.5 N-Step Tree Backup

📖 **Referans:** Sutton & Barto, Sayfa 158-160, Section 7.5

> *"In tree-backup updates, the target includes all rewards plus the estimated values of the dangling action nodes hanging off the sides"* (s. 158)

Importance sampling olmadan off-policy learning:

**Fikir**: Sadece alınan aksiyonları değil, **tüm aksiyonların değerlerini** kullan.

### Tree Backup Return - Equation 7.16 (s. 159)

$$G_{t:t+n} = R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1}) Q_{t+n-1}(S_{t+1}, a) + \gamma \pi(A_{t+1}|S_{t+1}) G_{t+1:t+n}$$

> *"The target of the one-step tree-backup update is the same as that of Expected Sarsa"* (s. 159)

Bu, bir **ağaç yapısı** oluşturur: alınan aksiyonun branch'ı devam eder, diğerleri "backup" edilir.

In [ ]:
def n_step_tree_backup(env, n, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    """
    N-step Tree Backup Algorithm.
    Referans: Algorithm (s. 160) - "n-step Tree Backup for estimating Q ≈ q_*"
    """
    # "Initialize Q(s, a) arbitrarily, for all s ∈ S, a ∈ A"
    Q = defaultdict(lambda: np.zeros(env.n_actions))
    
    def policy_probs(state):
        """
        ε-greedy action probabilities π(a|s).
        "the probability of taking a under ε-greedy policy"
        """
        probs = np.ones(env.n_actions) * epsilon / env.n_actions
        best_action = np.argmax(Q[state])
        probs[best_action] += 1 - epsilon
        return probs
    
    def select_action(state):
        probs = policy_probs(state)
        return np.random.choice(env.n_actions, p=probs)
    
    episode_lengths = []
    
    # "Loop for each episode:"
    for episode in range(n_episodes):
        states = [env.reset()]
        actions = [select_action(states[0])]
        rewards = [0]
        
        T = float('inf')
        t = 0
        
        while True:
            if t < T:
                next_state, reward, done = env.step(actions[t])
                states.append(next_state)
                rewards.append(reward)
                
                if done:
                    T = t + 1
                else:
                    actions.append(select_action(next_state))
            
            tau = t - n + 1
            
            if tau >= 0:
                # Tree backup calculation - Equation 7.16
                if t + 1 >= T:
                    G = rewards[T]
                else:
                    # "the leaves of the tree are the estimated action values"
                    probs = policy_probs(states[t + 1])
                    G = rewards[t + 1] + gamma * np.dot(probs, Q[states[t + 1]])
                
                # "working backward from state S_{t+1}"
                for k in range(min(t, T - 1), tau, -1):
                    probs = policy_probs(states[k])
                    # Sum over non-taken actions + taken action's G
                    G = rewards[k] + gamma * np.sum(
                        probs * Q[states[k]] * (np.arange(env.n_actions) != actions[k])
                    ) + gamma * probs[actions[k]] * G
                
                # "Q(S_τ, A_τ) ← Q(S_τ, A_τ) + α[G − Q(S_τ, A_τ)]"
                s, a = states[tau], actions[tau]
                Q[s][a] += alpha * (G - Q[s][a])
            
            if tau == T - 1:
                break
            
            t += 1
        
        episode_lengths.append(T)
    
    return Q, episode_lengths

Q_tree, lengths_tree = n_step_tree_backup(env_grid, n=4, n_episodes=200)
print(f"Tree Backup: Avg length = {np.mean(lengths_tree[-50:]):.1f}")

In [ ]:
# Compare all n-step methods
methods = {
    'n-step SARSA': lambda: n_step_sarsa(env_grid, n=4, n_episodes=200)[1],
    'Off-policy': lambda: n_step_off_policy_sarsa(env_grid, n=4, n_episodes=200)[1],
    'Tree Backup': lambda: n_step_tree_backup(env_grid, n=4, n_episodes=200)[1]
}

plt.figure(figsize=(10, 5))
window = 10

for name, method in methods.items():
    lengths = method()
    smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
    plt.plot(smoothed, label=name, linewidth=2)

plt.xlabel('Episode')
plt.ylabel('Steps per Episode')
plt.title('N-Step Methods Comparison (n=4)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Özet

📖 **Chapter 7 Key Points (s. 147-162)**

| Yöntem | n | Bootstrap | Variance | Bias |
|--------|---|-----------|----------|------|
| TD(0) | 1 | Yüksek | Düşük | Yüksek |
| n-step TD | n | Orta | Orta | Orta |
| MC | ∞ | Yok | Yüksek | Düşük |

> *"n-step methods span a spectrum from TD to MC, with intermediate n values often working better than either extreme."* (s. 147)

### N-Step Yöntemleri

| Yöntem | Sayfa | Tip | Özellik |
|--------|-------|-----|--------|
| **n-step TD** | s. 149-152 | Prediction | Basit n-step return |
| **n-step SARSA** | s. 153-155 | On-policy Control | Q için n-step |
| **n-step Off-policy** | s. 155-158 | Off-policy | Importance sampling gerekli |
| **Tree Backup** | s. 158-160 | Off-policy | IS gereksiz, daha kararlı |

### Anahtar Denklemler

**N-Step Return** (Eq. 7.1, s. 148):
$$G_{t:t+n} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V_{t+n-1}(S_{t+n})$$

**N-Step Sarsa** (Eq. 7.4, s. 153):
$$G_{t:t+n} = R_{t+1} + ... + \gamma^{n-1} R_{t+n} + \gamma^n Q_{t+n-1}(S_{t+n}, A_{t+n})$$

**Tree Backup** (Eq. 7.16, s. 159):
$$G_{t:t+n} = R_{t+1} + \gamma \sum_{a \neq A_{t+1}} \pi(a|S_{t+1}) Q(S_{t+1}, a) + \gamma \pi(A_{t+1}|S_{t+1}) G_{t+1:t+n}$$

---
### Sonraki Notebook
**08 - Planning and Learning** *(Chapter 8, s. 163-188)*: Model-based methods, Dyna-Q